# Timonel SFT - Banco A2 (Fase 1a, rev. 2)Objetivo: verificar si Llama 3.1 8B aplica el criterio de finalizacion`d < T -> terminar` y generaliza a valores de T no vistos.Diferencias con A1:- Prompt sin mencion a velocidad.- Respuesta en palabra (ir_a / terminar), no en digito.- Enmascaramiento del prompt verificado antes de entrenar.- 11 valores de T en train (no 4).

## 1. Instalar dependencias

In [ ]:
!pip install -q unsloth!pip install -q --no-deps trl peft accelerate bitsandbytes!pip install -q datasets

## 2. Cargar modelo base

In [ ]:
from unsloth import FastLanguageModelimport torchmax_seq_length = 1024model, tokenizer = FastLanguageModel.from_pretrained(    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",    max_seq_length = max_seq_length,    load_in_4bit = True,)

## 3. LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r = 16,    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",                      "gate_proj", "up_proj", "down_proj"],    lora_alpha = 32,    lora_dropout = 0.0,    bias = "none",    use_gradient_checkpointing = "unsloth",    random_state = 3407,)

## 4. Cargar banco A2

In [ ]:
import glob, osfrom datasets import load_dataset# Descubrimiento dinamico de rutas. Kaggle puede montar el dataset en# /kaggle/input/<nombre>/ o /kaggle/input/datasets/<user>/<nombre>/._candidatos = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)assert _candidatos, "No encontre train.jsonl en /kaggle/input"TRAIN_PATH = _candidatos[0]VAL_PATH = os.path.join(os.path.dirname(TRAIN_PATH), "val.jsonl")assert os.path.exists(VAL_PATH), f"No encontre val.jsonl en {VAL_PATH}"print(f"TRAIN_PATH = {TRAIN_PATH}")print(f"VAL_PATH   = {VAL_PATH}")dataset_train = load_dataset("json", data_files=TRAIN_PATH, split="train")dataset_val = load_dataset("json", data_files=VAL_PATH, split="train")print(f"Train: {len(dataset_train)} | Val: {len(dataset_val)}")print()print("Ejemplo 0 (prompt):")print(dataset_train[0]["prompt"][:400])print()print(f"T={dataset_train[0]['T']} d={dataset_train[0]['d']} label={dataset_train[0]['label']}")

## 5. Chat template + formateoLa respuesta esperada es un JSON con la accion como palabra:`{"accion": "ir_a"}` o `{"accion": "terminar"}`.

In [ ]:
from unsloth import get_chat_templatetokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")SYSTEM_PROMPT = ("Sos el timonel de un velero autonomo. Respondes "                 "UNICAMENTE con el JSON pedido, sin texto adicional.")def formatear(ejemplo):    respuesta = '{"accion": "' + ejemplo["label"] + '"}'    messages = [        {"role": "system", "content": SYSTEM_PROMPT},        {"role": "user", "content": ejemplo["prompt"]},        {"role": "assistant", "content": respuesta},    ]    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}dataset_train = dataset_train.map(formatear)dataset_val = dataset_val.map(formatear)print("--- Ejemplo formateado (primeros 800 chars) ---")print(dataset_train[0]["text"][:800])

## 6. SFTTrainerSe crea el trainer con la config conservadora (misma que A1).El enmascaramiento se aplica DESPUES de crear el trainer y severifica ANTES de train().

In [ ]:
from trl import SFTTrainerfrom transformers import TrainingArgumentsfrom unsloth import is_bfloat16_supportedtrainer = SFTTrainer(    model = model, tokenizer = tokenizer,    train_dataset = dataset_train, eval_dataset = dataset_val,    dataset_text_field = "text",    max_seq_length = max_seq_length,    dataset_num_proc = 2, packing = False,    args = TrainingArguments(        per_device_train_batch_size = 2,        gradient_accumulation_steps = 4,        warmup_steps = 10, max_steps = 154,        learning_rate = 2e-4,        fp16 = not is_bfloat16_supported(),        bf16 = is_bfloat16_supported(),        logging_steps = 10,        eval_strategy = "steps", eval_steps = 50,        save_strategy = "steps", save_steps = 50,        optim = "adamw_8bit", weight_decay = 0.01,        lr_scheduler_type = "linear",        seed = 3407, output_dir = "outputs_A2", report_to = "none",    ),)

## 7. Enmascaramiento del promptAplica train_on_responses_only para que el loss se calcule SOLOsobre la respuesta del asistente. Esto es lo que no se hizo en A1.

In [ ]:
from unsloth.chat_templates import train_on_responses_onlytrainer = train_on_responses_only(    trainer,    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",)print("[OK] train_on_responses_only aplicado")

## 8. Verificacion de enmascaramientoCompuerta tecnica: si el enmascaramiento fallo, NO se entrena.Se inspeccionan 3 ejemplos y se verifica que los tokens supervisadossean pocos (la respuesta JSON, ~10 tokens) y NO incluyan el prompt.

In [ ]:
import mathdef inspect_masking(trainer, tokenizer, max_sup_tokens=30):    batch = next(iter(trainer.get_train_dataloader()))    ids, labels = batch["input_ids"], batch["labels"]    ok = True    n_sup_list = []    for i in range(min(3, ids.shape[0])):        sup = [int(t) for t, l in zip(ids[i], labels[i]) if int(l) != -100]        txt = tokenizer.decode(sup)        n_sup_list.append(len(sup))        print(f"--- ej {i}: {len(sup)} tokens supervisados de {ids.shape[1]}")        print("SUPERVISADO:", repr(txt))        if not (0 < len(sup) <= max_sup_tokens) or "PROPOSITO" in txt:            ok = False    if n_sup_list and n_sup_list[0] > 0:        print(f"Referencia: si la decision fuera azar, loss medio ~ ln2/N = {math.log(2)/n_sup_list[0]:.4f}")    print("ENMASCARAMIENTO OK" if ok else "ENMASCARAMIENTO FALLIDO")    return okmasking_ok = inspect_masking(trainer, tokenizer)assert masking_ok, "No entrenar: corregir train_on_responses_only"print()print("[COMPUERTA] Enmascaramiento: PASO")

## 9. Entrenamiento

In [ ]:
stats = trainer.train()print(stats)

## 10. Guardar

In [ ]:
model.save_pretrained("timonel_lora_banco_A2")tokenizer.save_pretrained("timonel_lora_banco_A2")!ls -la timonel_lora_banco_A2/